# Bot clássico ELIZA — do código à interação
Aprenda a construir Bot clássico ELIZA com o mesmo código Python do projeto interativo. A sequência ensina estado, regras e interface. Cada trecho é parte do programa final; os recuos devem ser mantidos. A instalação vai em uma célula separada e o programa completo em outra.

In [ ]:
%pip install -q "ipywidgets>=8.1,<9"


## Normalizar e refletir as palavras
re encontra padrões; unicodedata ajuda a comparar mensagens sem diferença de acentos ou maiúsculas. refletir troca palavras completas: minha vira sua e eu vira você. re.sub faz uma única passagem para não trocar de volta palavras recém-alteradas. Este é um trecho do programa completo: acrescente depois da parte anterior, preservando os espaços. Não execute este fragmento isoladamente.

```python
import re
import unicodedata

def normalizar(texto):
    return "".join(c for c in unicodedata.normalize("NFD", texto.lower()) if not unicodedata.combining(c))

def refletir(texto):
    trocas = {"eu": "você", "meu": "seu", "minha": "sua", "meus": "seus", "minhas": "suas",
              "sou": "é", "estou": "está", "tenho": "tem", "voce": "eu", "seu": "meu", "sua": "minha"}
    return re.sub(r"\b\w+\b", lambda m: trocas.get(normalizar(m.group()), m.group()), texto)


```

Por que trocar palavras completas em vez de partes das palavras?

## Criar uma sessão de conversa
A classe guarda se a conversa acabou e um contador para alternar respostas genéricas. Cada Nova conversa cria uma instância com estado novo. Este é um trecho do programa completo: acrescente depois da parte anterior, preservando os espaços. Não execute este fragmento isoladamente.

```python
class Eliza:
    def __init__(self):
        self.encerrada = False
        self.turno = 0


```

O que deve ser apagado quando uma nova conversa começa?

## Reconhecer frases e construir respostas
responder trata mensagem vazia, despedida e saudação. Depois procura padrões como eu estou e eu preciso. O grupo (.+) captura o restante da frase; refletir adapta pronomes antes de montar a resposta. Regras de família e programação vêm depois; por último alternamos respostas genéricas. Este é um trecho do programa completo: acrescente depois da parte anterior, preservando os espaços. Não execute este fragmento isoladamente.

```python
    def responder(self, texto):
        if self.encerrada:
            return "Conversa encerrada. Clique em Nova conversa."
        texto = texto.strip()
        if not texto:
            return "Escreva uma frase para começarmos."
        n = normalizar(texto)
        if n in ["sair", "tchau", "adeus"]:
            self.encerrada = True
            return "Até mais! Obrigada pela conversa."
        if re.search(r"\b(oi|ola)\b", n):
            return "Olá! Sou uma ELIZA didática, um programa de regras. Sobre o que quer conversar?"
        for padrao, resposta in [
            (r"\beu preciso (.+)", "Por que você precisa {trecho}?"),
            (r"\beu sinto (.+)", "O que faz você sentir {trecho}?"),
            (r"\beu estou (.+)", "Há quanto tempo você está {trecho}?"),
            (r"\beu sou (.+)", "O que significa para você ser {trecho}?"),
            (r"\bporque (.+)", "Essa é a única razão: {trecho}?"),
        ]:
            achado = re.search(padrao, n)
            if achado:
                # A normalização mantém os índices das letras portuguesas usuais.
                trecho = texto[achado.start(1):achado.end(1)].rstrip(".!?")
                trecho = refletir(trecho)
                if "precisa {trecho}" in resposta and not re.match(r"^(de|do|da|dos|das)\b", normalizar(trecho)):
                    trecho = "de " + trecho
                return resposta.format(trecho=trecho)
        if re.search(r"\b(mae|pai|familia|irmao|irma)\b", n):
            return "Pode me contar mais sobre sua família?"
        if re.search(r"\b(python|programacao|codigo)\b", n):
            return "Que parte da programação você está aprendendo?"
        respostas = ["Pode dar um exemplo?", "O que você pensa sobre isso?", "Conte um pouco mais."]
        resposta = respostas[self.turno % len(respostas)]
        self.turno += 1
        return resposta


```

Por que a resposta genérica deve ficar depois das regras específicas?

## Preparar a caixa de conversa
Criamos a instância Eliza, a lista de histórico, o painel HTML, a caixa de texto e os botões. O modelo é local e baseado em regras; não recebe treinamento nem usa API. Este é um trecho do programa completo: acrescente depois da parte anterior, preservando os espaços. Não execute este fragmento isoladamente.

```python
if __name__ == "__main__":
    import html
    import ipywidgets as w
    from IPython.display import display
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except ImportError:
        pass
    eliza = Eliza()
    historico_eliza = []
    conversa_eliza = w.HTML()
    entrada_eliza = w.Textarea(description="Você:", placeholder="Digite sua mensagem…", layout=w.Layout(width="95%"))
    enviar_eliza = w.Button(description="Enviar")
    nova_eliza = w.Button(description="Nova conversa")


```

Qual é a diferença entre o estado de Eliza e a lista historico_eliza?

## Mostrar mensagens com segurança
atualizar_eliza transforma as últimas mensagens em parágrafos e usa html.escape para mostrar a entrada como texto. O painel tem limite de altura e rolagem. Este é um trecho do programa completo: acrescente depois da parte anterior, preservando os espaços. Não execute este fragmento isoladamente.

```python
    def atualizar_eliza():
        conversa_eliza.value = '<h3>ELIZA didática</h3><p>Bot por regras inspirado em ELIZA. Não é uma LLM nem um serviço de aconselhamento.</p><div style="max-height:300px;overflow:auto">' + "".join(
            f'<p><b>{autor}:</b> {html.escape(texto)}</p>' for autor, texto in historico_eliza[-60:]) + '</div>'


```

Por que uma mensagem contendo <script> deve aparecer como texto?

## Enviar e reiniciar a conversa
enviar_mensagem lê o texto, registra usuário e resposta, limpa a caixa e atualiza a tela. nova_conversa cria outra Eliza e limpa o histórico. Cada operação é acionada por um botão. Este é um trecho do programa completo: acrescente depois da parte anterior, preservando os espaços. Não execute este fragmento isoladamente.

```python
    def enviar_mensagem(_):
        texto = entrada_eliza.value.strip()
        if not texto:
            return
        historico_eliza.append(("Você", texto))
        historico_eliza.append(("ELIZA", eliza.responder(texto)))
        entrada_eliza.value = ""
        atualizar_eliza()

    def nova_conversa(_):
        global eliza
        eliza = Eliza()
        historico_eliza.clear()
        historico_eliza.append(("ELIZA", "Olá! Digite uma frase. Para encerrar, escreva sair."))
        entrada_eliza.value = ""
        atualizar_eliza()


```

Por que atualizar a tela só depois de registrar a resposta?

## Ativar os eventos e apresentar o chat
on_click conecta os botões às funções. display coloca os widgets na saída; nova_conversa prepara a primeira mensagem. Digite uma frase e clique em Enviar; sair encerra a conversa. Este é um trecho do programa completo: acrescente depois da parte anterior, preservando os espaços. Não execute este fragmento isoladamente.

```python
    enviar_eliza.on_click(enviar_mensagem)
    nova_eliza.on_click(nova_conversa)
    display(w.VBox([conversa_eliza, entrada_eliza, w.HBox([enviar_eliza, nova_eliza])]))
    nova_conversa(None)

```

Qual função será chamada quando você clicar em Enviar?

## Executar o programa completo
Este é exatamente o programa montado nas etapas anteriores e usado no notebook vinculado. Depois da instalação, copie este bloco inteiro para outra célula e execute. Use os botões e a caixa de mensagem, quando disponível. Mantenha a sessão do Colab conectada.

In [ ]:
import re
import unicodedata

def normalizar(texto):
    return "".join(c for c in unicodedata.normalize("NFD", texto.lower()) if not unicodedata.combining(c))

def refletir(texto):
    trocas = {"eu": "você", "meu": "seu", "minha": "sua", "meus": "seus", "minhas": "suas",
              "sou": "é", "estou": "está", "tenho": "tem", "voce": "eu", "seu": "meu", "sua": "minha"}
    return re.sub(r"\b\w+\b", lambda m: trocas.get(normalizar(m.group()), m.group()), texto)

class Eliza:
    def __init__(self):
        self.encerrada = False
        self.turno = 0

    def responder(self, texto):
        if self.encerrada:
            return "Conversa encerrada. Clique em Nova conversa."
        texto = texto.strip()
        if not texto:
            return "Escreva uma frase para começarmos."
        n = normalizar(texto)
        if n in ["sair", "tchau", "adeus"]:
            self.encerrada = True
            return "Até mais! Obrigada pela conversa."
        if re.search(r"\b(oi|ola)\b", n):
            return "Olá! Sou uma ELIZA didática, um programa de regras. Sobre o que quer conversar?"
        for padrao, resposta in [
            (r"\beu preciso (.+)", "Por que você precisa {trecho}?"),
            (r"\beu sinto (.+)", "O que faz você sentir {trecho}?"),
            (r"\beu estou (.+)", "Há quanto tempo você está {trecho}?"),
            (r"\beu sou (.+)", "O que significa para você ser {trecho}?"),
            (r"\bporque (.+)", "Essa é a única razão: {trecho}?"),
        ]:
            achado = re.search(padrao, n)
            if achado:
                # A normalização mantém os índices das letras portuguesas usuais.
                trecho = texto[achado.start(1):achado.end(1)].rstrip(".!?")
                trecho = refletir(trecho)
                if "precisa {trecho}" in resposta and not re.match(r"^(de|do|da|dos|das)\b", normalizar(trecho)):
                    trecho = "de " + trecho
                return resposta.format(trecho=trecho)
        if re.search(r"\b(mae|pai|familia|irmao|irma)\b", n):
            return "Pode me contar mais sobre sua família?"
        if re.search(r"\b(python|programacao|codigo)\b", n):
            return "Que parte da programação você está aprendendo?"
        respostas = ["Pode dar um exemplo?", "O que você pensa sobre isso?", "Conte um pouco mais."]
        resposta = respostas[self.turno % len(respostas)]
        self.turno += 1
        return resposta

if __name__ == "__main__":
    import html
    import ipywidgets as w
    from IPython.display import display
    try:
        from google.colab import output
        output.enable_custom_widget_manager()
    except ImportError:
        pass
    eliza = Eliza()
    historico_eliza = []
    conversa_eliza = w.HTML()
    entrada_eliza = w.Textarea(description="Você:", placeholder="Digite sua mensagem…", layout=w.Layout(width="95%"))
    enviar_eliza = w.Button(description="Enviar")
    nova_eliza = w.Button(description="Nova conversa")

    def atualizar_eliza():
        conversa_eliza.value = '<h3>ELIZA didática</h3><p>Bot por regras inspirado em ELIZA. Não é uma LLM nem um serviço de aconselhamento.</p><div style="max-height:300px;overflow:auto">' + "".join(
            f'<p><b>{autor}:</b> {html.escape(texto)}</p>' for autor, texto in historico_eliza[-60:]) + '</div>'

    def enviar_mensagem(_):
        texto = entrada_eliza.value.strip()
        if not texto:
            return
        historico_eliza.append(("Você", texto))
        historico_eliza.append(("ELIZA", eliza.responder(texto)))
        entrada_eliza.value = ""
        atualizar_eliza()

    def nova_conversa(_):
        global eliza
        eliza = Eliza()
        historico_eliza.clear()
        historico_eliza.append(("ELIZA", "Olá! Digite uma frase. Para encerrar, escreva sair."))
        entrada_eliza.value = ""
        atualizar_eliza()

    enviar_eliza.on_click(enviar_mensagem)
    nova_eliza.on_click(nova_conversa)
    display(w.VBox([conversa_eliza, entrada_eliza, w.HBox([enviar_eliza, nova_eliza])]))
    nova_conversa(None)
